In [275]:
import fine as fn
from getData import getData
from pathlib import Path
import pandas as pd
import numpy as np


cwd = Path.cwd()
data = getData()

%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [276]:
locations = {"Region1", "Region2", "Region3"}
commodityUnitDict = {"electricity": r"GW$_{el}$", "hydrogen": r"GW$_{H2}$", "heat": r"GW$_{th}$"}
commodities = {"electricity", "hydrogen", "heat"}
numberOfTimeSteps = 4
hoursPerTimeStep = 1

In [277]:
esM = fn.EnergySystemModel(
    locations=locations,
    commodities=commodities,
    numberOfTimeSteps=4,
    commodityUnitsDict=commodityUnitDict,
    hoursPerTimeStep=1,
    costUnit="1e9 Euro",
    lengthUnit="km",
    verboseLogLevel=0,
)

In [278]:
windOperationRateMax = pd.DataFrame({
    "Region1": [
        1,
        1,
        1,
        1,
    ],
    "Region2": [
        1,
        1,
        1,
        1,
    ],
    "Region3": [
        1,
        1,
        0,
        0,
    ]
})

print(windOperationRateMax)

esM.add(
    fn.Source(
        esM=esM,
        name="Wind",
        commodity="electricity",
        hasCapacityVariable=True,
        capacityMax=1,
        operationRateMax=windOperationRateMax,
        investPerCapacity=2 * 2190,
        opexPerCapacity=0,
        interestRate=0,
        opexPerOperation=0,
        economicLifetime=1,
    )
)

   Region1  Region2  Region3
0        1        1        1
1        1        1        1
2        1        1        0
3        1        1        0


In [279]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="Electrolyzer",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": -1, "hydrogen": 0.7},
        hasCapacityVariable=True,
        investPerCapacity=0.5,
        opexPerCapacity=0.5 * 0.025,
        interestRate=0.08,
        economicLifetime=10,
    )
)

In [280]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Li-ion batteries",
        commodity="electricity",
        hasCapacityVariable=True,
        capacityMax=1,
        chargeEfficiency=0.95,
        cyclicLifetime=10000,
        dischargeEfficiency=0.95,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        doPreciseTsaModeling=False,
        investPerCapacity=0.151,
        opexPerCapacity=0.002,
        interestRate=0.08,
        economicLifetime=22,
    )
)

In [281]:
distances = np.array(
    [
        [0, 1, 1],
        [1, 0, 1],
        [1, 1, 0]
    ]
)


incidence = np.array(
    [
        [0, 1, 1],
        [1, 0, 1],
        [1, 1, 0]
    ]
)

# incidence = np.array(
#     [
#         [0, 0, 0],
#         [0, 0, 0],
#         [0, 0, 0]
#     ]
# )

locs = sorted(locations)   # oder list(locations)

distances = pd.DataFrame(distances, index=locs, columns=locs)
eligibility = pd.DataFrame(incidence, index=locs, columns=locs)

esM.add(
    fn.Transmission(
        esM=esM,
        name="AC cable",
        commodity="electricity",
        losses=0,
        distances=distances,
        hasCapacityVariable=True,
        capacityMax=1,
        locationalEligibility=eligibility,
        investPerCapacity=0.1,
        interestRate=0.08,
        economicLifetime=50,
    )
)

In [282]:
distances = np.array(
    [
        [0, 1, 1],
        [1, 0, 1],
        [1, 1, 0]
    ]
)


incidence = np.array(
    [
        [0, 1, 1],
        [1, 0, 1],
        [1, 1, 0]
    ]
)

# incidence = np.array(
#     [
#         [0, 0, 0],
#         [0, 0, 0],
#         [0, 0, 0]
#     ]
# )

locs = sorted(locations)   # oder list(locations)

distances = pd.DataFrame(distances, index=locs, columns=locs)
eligibility = pd.DataFrame(incidence, index=locs, columns=locs)

esM.add(
    fn.Transmission(
        esM=esM,
        name="hydrogen pipeline",
        commodity="hydrogen",
        losses=0,
        distances=distances,
        capacityMax=1,
        hasCapacityVariable=True,
        locationalEligibility=eligibility,
        investPerCapacity=0.1,
        interestRate=0.08,
        economicLifetime=50,
    )
)

In [283]:
Demand = pd.DataFrame({
    "Region1": [
        1,
        1,
        1,
        1,
    ],
    "Region2": [
        0,
        0,
        0,
        0,
    ],
    "Region3": [
        0,
        0,
        0,
        4,
    ],
})

esM.add(
    fn.Sink(
        esM=esM,
        name="Hydrogen demand",
        commodity="hydrogen",
        #commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=Demand,
    )
)

# esM.add(
#     fn.Sink(
#         esM=esM,
#         name="CO2 demand",
#         commodity="CO2",
#         #commodity="electricity",
#         hasCapacityVariable=False,
#         operationRateFix=Demand,
#     )
# )

#esM.declareOptimizationProblem()

In [284]:
# import pandas as pd


# def _region_has_positive_demand(sink, region):
#     """
#     Prüft, ob ein Sink in einer Region einen positiven Demand hat.
#     Relevant sind typischerweise operationRateFix oder operationRateMin.
#     """

#     for attr in ["operationRateFix", "operationRateMin"]:
#         value = getattr(sink, attr, None)

#         if value is None:
#             continue

#         # Zeitreihe als DataFrame
#         if isinstance(value, pd.DataFrame):
#             if region in value.columns and (value[region] > 0).any():
#                 return True

#         # Regionaler Series-Demand
#         elif isinstance(value, pd.Series):
#             if region in value.index and value.loc[region] > 0:
#                 return True

#         # Skalarer Demand
#         elif isinstance(value, (int, float)):
#             if value > 0:
#                 return True

#     return False


# def check_sink_commodities_can_be_supplied(esM):
#     """
#     Prüft:
#     Wenn ein Sink in einer Region positive Nachfrage nach Commodity X hat,
#     muss es in derselben Region entweder
#     - eine Source für Commodity X geben oder
#     - eine Conversion geben, die Commodity X erzeugt.
#     """

#     errors = []

#     supplied_commodities_by_region = {
#         loc: set() for loc in esM.locations
#     }

#     # -------------------------
#     # Sources und Conversions sammeln
#     # -------------------------
#     for model in esM.componentModelingDict.values():
#         for comp_name, comp in model.componentsDict.items():

#             comp_type = comp.__class__.__name__

#             # Source stellt ihre commodity bereit
#             if comp_type == "Source":
#                 commodity = comp.commodity

#                 for loc in comp.processedLocationalEligibility.index:
#                     if comp.processedLocationalEligibility.loc[loc] == 1:
#                         supplied_commodities_by_region[loc].add(commodity)

#             # Conversion stellt Commodities mit positivem Faktor bereit
#             elif comp_type == "Conversion":
#                 for commodity, factor in comp.commodityConversionFactors.items():
#                     if factor > 0:
#                         for loc in comp.processedLocationalEligibility.index:
#                             if comp.processedLocationalEligibility.loc[loc] == 1:
#                                 supplied_commodities_by_region[loc].add(commodity)

#             # elif comp_type == "Transmission":
#             #     for commodity, factor in comp.commodityConversionFactors.items():
#             #         if factor > 0:
#             #             for loc in comp.processedLocationalEligibility.index:
#             #                 if comp.processedLocationalEligibility.loc[loc] == 1:
#             #                     supplied_commodities_by_region[loc].add(commodity)

#     # -------------------------
#     # Sinks prüfen
#     # -------------------------
#     for model in esM.componentModelingDict.values():
#         for comp_name, comp in model.componentsDict.items():

#             if comp.__class__.__name__ != "Sink":
#                 continue

#             sink_commodity = comp.commodity

#             for loc in esM.locations:
#                 if not _region_has_positive_demand(comp, loc):
#                     continue

#                 if sink_commodity not in supplied_commodities_by_region[loc]:
#                     errors.append(
#                         f"Sink '{comp_name}' has positive demand for commodity "
#                         f"'{sink_commodity}' in region '{loc}', but this commodity "
#                         f"cannot be supplied there by any Source or Conversion."
#                     )

#     if errors:
#         raise ValueError(
#             "Commodity supply check failed:\n\n" + "\n".join(errors)
#         )

#     print("Commodity supply check passed.")

# check_sink_commodities_can_be_supplied(esM)

In [285]:
'''
First test:
Checks if all regions are connected or there is an island region
'''
import networkx as nx
 
def check_transmission_connectivity(esM):
    tm = esM.componentModelingDict.get("TransmissionModel")
    problems = []
    if tm is None:
        return problems
 
    by_commodity = {}
    for comp in tm.componentsDict.values():
        by_commodity.setdefault(comp.commodity, []).append(comp)
 
    for commodity, comps in by_commodity.items():
        G = nx.Graph()
        G.add_nodes_from(esM.locations)
        for comp in comps:
            for edge_key, eligible in comp.locationalEligibility.items():
                if eligible > 0:
                    loc1, loc2 = _parse_edge_key(edge_key)
                    G.add_edge(loc1, loc2)
 
        if not nx.is_connected(G):
            components = list(nx.connected_components(G))
            problems.append(
                f"Commodity '{commodity}': Transmission-Netz ist nicht "
                f"vollständig verbunden. Isolierte Gruppen: {components}"
            )
    return problems

#check_transmission_connectivity(esM)

In [286]:
'''
Second test:
Checks if all sink commodities can be produced from the available source commodities and conversion components
'''
def check_sink_commodities_producible(esM, raise_error=True):
    """
    Prüft, ob alle Sink-Commodities aus Sources und Conversion-Komponenten
    erzeugt werden können.

    Wichtig:
    Eine Conversion darf nur verwendet werden, wenn ALLE Input-Commodities
    dieser Conversion bereits verfügbar sind.
    """

    ssm = esM.componentModelingDict.get("SourceSinkModel")
    conv_model = esM.componentModelingDict.get("ConversionModel")

    source_commodities = set()
    sink_commodities = set()

    # -------------------------
    # Sources und Sinks sammeln
    # -------------------------
    if ssm is not None:
        for comp in ssm.componentsDict.values():
            if getattr(comp, "sign", None) == 1:
                source_commodities.add(comp.commodity)

            elif getattr(comp, "sign", None) == -1:
                sink_commodities.add(comp.commodity)

    producible_commodities = set(source_commodities)

    # -------------------------
    # Conversions iterativ anwenden
    # -------------------------
    changed = True

    while changed:
        changed = False

        if conv_model is None:
            break

        for comp in conv_model.componentsDict.values():
            factors = comp.commodityConversionFactors

            input_commodities = {
                com for com, factor in factors.items()
                if factor < 0
            }

            output_commodities = {
                com for com, factor in factors.items()
                if factor > 0
            }

            # Conversion ist nur nutzbar, wenn ALLE Inputs verfügbar sind
            if input_commodities.issubset(producible_commodities):
                before = len(producible_commodities)
                producible_commodities.update(output_commodities)

                if len(producible_commodities) > before:
                    changed = True

    missing_sink_commodities = sink_commodities - producible_commodities

    if missing_sink_commodities:
        message = (
            "The following sink commodities cannot be produced from the "
            "available source commodities and conversion components:\n"
            + "\n".join(f"- {com}" for com in sorted(missing_sink_commodities))
            + "\n\nAvailable source commodities:\n"
            + "\n".join(f"- {com}" for com in sorted(source_commodities))
            + "\n\nProducible commodities:\n"
            + "\n".join(f"- {com}" for com in sorted(producible_commodities))
        )

        if raise_error:
            raise ValueError(message)

        return missing_sink_commodities

    return set()

#check_sink_commodities_producible(esM)

In [287]:
import pandas as pd


def _eligible_locations(comp, esM):
    """Locations, an denen die Komponente gebaut/betrieben werden kann.
    None bei locationalEligibility bedeutet in FINE: überall erlaubt."""
    elig = getattr(comp, "processedLocationalEligibility", None)
    if elig is None:
        elig = getattr(comp, "locationalEligibility", None)
    if elig is not None:
        return {loc for loc, v in elig.items() if v > 0}
    return set(esM.locations)


def _parse_edge(edge_key, locations):
    """Zerlegt einen Transmission-Edge-Key 'loc1_loc2' robust,
    auch wenn Regionsnamen selbst Unterstriche enthalten."""
    for loc in sorted(locations, key=len, reverse=True):
        if edge_key.startswith(loc + "_"):
            rest = edge_key[len(loc) + 1:]
            if rest in locations:
                return loc, rest
    raise ValueError(f"Kann Edge-Key '{edge_key}' nicht parsen.")


def _get_time_series(comp, base_name):
    """Holt eine Zeitreihe unabhängig von der FINE-Version
    (z.B. operationRateFix / fullOperationRateFix / processedOperationRateFix)."""
    for attr in (f"processed{base_name[0].upper()}{base_name[1:]}",
                 f"full{base_name[0].upper()}{base_name[1:]}",
                 base_name):
        ts = getattr(comp, attr, None)
        if isinstance(ts, pd.DataFrame):
            return ts
    return None


def _has_positive_supply(comp, loc):
    """False, wenn eine Source an der Location per operationRateMax/Fix
    effektiv nichts liefern kann (z.B. Wind mit Nullzeitreihe)."""
    for base in ("operationRateMax", "operationRateFix"):
        rate = _get_time_series(comp, base)
        if rate is not None and loc in rate.columns:
            return rate[loc].sum() > 0
    return True


def check_commodity_reachability(esM):
    """Prüft für jede Sink-Commodity mit Nachfrage, ob sie in der jeweiligen
    Region produzierbar oder importierbar ist — direkt, über
    Conversion-Ketten oder Transmission.

    Gibt eine Liste von Problem-Strings zurück (leer = Check bestanden).
    """
    problems = []

    srcSnkModel = esM.componentModelingDict.get("SourceSinkModel")
    convModel = esM.componentModelingDict.get("ConversionModel")
    transModel = esM.componentModelingDict.get("TransmissionModel")

    # --- Sources und Sinks trennen (Sink erbt in FINE von Source → sign) ---
    sources, sinks = [], []
    if srcSnkModel:
        for comp in srcSnkModel.componentsDict.values():
            (sources if comp.sign == 1 else sinks).append(comp)

    # --- Startmenge: (commodity, region), die direkt produzierbar sind ---
    available = set()
    for comp in sources:
        for loc in _eligible_locations(comp, esM):
            if _has_positive_supply(comp, loc):
                available.add((comp.commodity, loc))

    # --- Transmission-Kanten einmalig einsammeln ---
    trans_edges = []  # Liste von (commodity, loc1, loc2)
    if transModel:
        for comp in transModel.componentsDict.values():
            elig = getattr(comp, "processedLocationalEligibility", None)
            if elig is None:
                elig = getattr(comp, "locationalEligibility", None)
            if elig is not None:
                pairs = [_parse_edge(e, esM.locations)
                         for e, v in elig.items() if v > 0]
            else:
                locs = list(esM.locations)
                pairs = [(a, b) for a in locs for b in locs if a != b]
            for loc1, loc2 in pairs:
                # beide Richtungen, Transmission ist bidirektional nutzbar
                trans_edges.append((comp.commodity, loc1, loc2))
                trans_edges.append((comp.commodity, loc2, loc1))

    # --- Fixpunkt-Iteration: Conversions + Transmission ---
    changed = True
    while changed:
        changed = False

        if convModel:
            for comp in convModel.componentsDict.values():
                inputs = {c for c, f in comp.commodityConversionFactors.items() if f < 0}
                outputs = {c for c, f in comp.commodityConversionFactors.items() if f > 0}
                for loc in _eligible_locations(comp, esM):
                    if all((c, loc) in available for c in inputs):
                        new = {(c, loc) for c in outputs} - available
                        if new:
                            available |= new
                            changed = True

        for commodity, loc1, loc2 in trans_edges:
            if (commodity, loc1) in available and (commodity, loc2) not in available:
                available.add((commodity, loc2))
                changed = True

    # --- Sinks prüfen ---
    for snk in sinks:
        demand = _get_time_series(snk, "operationRateFix")
        if demand is None:
            continue
        for loc in demand.columns:
            if demand[loc].sum() > 0 and (snk.commodity, loc) not in available:
                problems.append(
                    f"Sink '{snk.name}': Nachfrage nach '{snk.commodity}' in "
                    f"'{loc}', aber die Commodity ist dort weder produzierbar "
                    f"noch importierbar."
                )

    return problems


check_commodity_reachability(esM)

[]

In [288]:
import math
from collections import defaultdict

import pandas as pd


# ---------------------------------------------------------------------------
# Helper
# ---------------------------------------------------------------------------

def _unwrap_ip(obj):
    """FINE >= 2.x speichert processed-Attribute als dict
    {investmentPeriod: Series/DataFrame}. Nimmt die erste Periode."""
    if isinstance(obj, dict):
        if not obj:
            return None
        return obj[sorted(obj.keys())[0]]
    return obj


def _get_time_series(comp, base_name):
    """Holt eine Zeitreihe unabhängig von der FINE-Version
    (processed... / full... / Roh-Attribut)."""
    for attr in (f"processed{base_name[0].upper()}{base_name[1:]}",
                 f"full{base_name[0].upper()}{base_name[1:]}",
                 base_name):
        ts = _unwrap_ip(getattr(comp, attr, None))
        if isinstance(ts, pd.DataFrame):
            return ts
    return None


def _eligible_locations(comp, esM):
    """Locations, an denen die Komponente gebaut/betrieben werden kann.
    None bei locationalEligibility bedeutet in FINE: überall erlaubt."""
    elig = _unwrap_ip(getattr(comp, "processedLocationalEligibility", None))
    if elig is None:
        elig = _unwrap_ip(getattr(comp, "locationalEligibility", None))
    if elig is not None:
        return {loc for loc, v in elig.items() if v > 0}
    return set(esM.locations)


def _capacity_series(comp, esM):
    """Obere Kapazitätsschranke je Location. NaN/None -> unbegrenzt (inf)."""
    for attr in ("processedCapacityFix", "capacityFix",
                 "processedCapacityMax", "capacityMax"):
        cap = _unwrap_ip(getattr(comp, attr, None))
        if cap is None:
            continue
        if isinstance(cap, pd.Series):
            return cap.astype(float).fillna(math.inf)
        return pd.Series(float(cap),
                         index=sorted(_eligible_locations(comp, esM)))
    return pd.Series(math.inf,
                     index=sorted(_eligible_locations(comp, esM)))


def _source_max_energy(comp, esM):
    """Maximal einspeisbare Energie einer Source über den Horizont,
    summiert über alle Locations (in commodityUnit * h)."""
    h = esM.hoursPerTimeStep
    T = getattr(esM, "numberOfTimeSteps", None) or len(esM.totalTimeSteps)

    rate = _get_time_series(comp, "operationRateFix")
    if rate is None:
        rate = _get_time_series(comp, "operationRateMax")

    caps = _capacity_series(comp, esM)
    total = 0.0
    for loc in _eligible_locations(comp, esM):
        if comp.hasCapacityVariable:
            cap = float(caps.get(loc, math.inf))
            # Zeitreihe ist relativ zur Kapazität
            r = rate[loc].sum() if (rate is not None and loc in rate.columns) else T
            total += 0.0 if r == 0 else cap * r * h
        else:
            # Zeitreihe ist absolut in Commodity-Einheiten
            if rate is not None and loc in rate.columns:
                total += rate[loc].sum()
            else:
                total += math.inf
    return total


# ---------------------------------------------------------------------------
# Test 2: aggregierte Commodity-Bilanz
# ---------------------------------------------------------------------------

def check_commodity_balance(esM, tol=1e-6, max_iter=100):
    """Aggregierte Mengenbilanz (nicht zeit-, nicht regionsaufgelöst):
    Kann über Sources + Conversion-Ketten genug von jeder Commodity
    bereitgestellt werden, um die fixe Nachfrage der Sinks zu decken?

    Obere Schranke -> notwendige, nicht hinreichende Bedingung
    (ignoriert Transmission-Engpässe, Input-Konkurrenz zwischen
    Conversions und Speicherverluste).

    Gibt (problems, report) zurück; problems leer = Check bestanden.
    """
    problems = []
    h = esM.hoursPerTimeStep
    T = getattr(esM, "numberOfTimeSteps", None) or len(esM.totalTimeSteps)

    srcSnkModel = esM.componentModelingDict.get("SourceSinkModel")
    convModel = esM.componentModelingDict.get("ConversionModel")

    # --- Sources und Sinks trennen (Sink erbt in FINE von Source -> sign) ---
    sources, sinks = [], []
    if srcSnkModel:
        for comp in srcSnkModel.componentsDict.values():
            (sources if comp.sign == 1 else sinks).append(comp)
    conversions = list(convModel.componentsDict.values()) if convModel else []

    # --- direkte Einspeisung je Commodity ---
    direct_supply = defaultdict(float)
    for comp in sources:
        direct_supply[comp.commodity] += _source_max_energy(comp, esM)

    # --- fixe Nachfrage je Commodity ---
    total_demand = defaultdict(float)
    for snk in sinks:
        dem = _get_time_series(snk, "operationRateFix")
        if dem is not None:
            total_demand[snk.commodity] += float(dem.sum().sum())

    # --- Fixpunkt: verfügbare Menge inkl. Conversion-Ketten ---
    avail = defaultdict(float, direct_supply)
    for _ in range(max_iter):
        new_avail = defaultdict(float, direct_supply)
        for comp in conversions:
            factors = comp.commodityConversionFactors
            factors = _unwrap_ip(factors) if isinstance(factors, dict) and all(
                isinstance(k, int) for k in factors
            ) else factors
            inputs = {c: abs(f) for c, f in factors.items() if f < 0}
            outputs = {c: f for c, f in factors.items() if f > 0}
            # max. Betrieb in physicalUnit*h: Kapazität und Input-Verfügbarkeit
            cap_energy = float(_capacity_series(comp, esM).sum())
            op = math.inf if math.isinf(cap_energy) else cap_energy * T * h
            for c_in, f_in in inputs.items():
                op = min(op, avail[c_in] / f_in)
            for c_out, f_out in outputs.items():
                new_avail[c_out] += op * f_out
        # Konvergenz prüfen
        keys = set(avail) | set(new_avail)
        if all(math.isclose(avail[k], new_avail[k], rel_tol=1e-9, abs_tol=tol)
               or (math.isinf(avail[k]) and math.isinf(new_avail[k]))
               for k in keys):
            avail = new_avail
            break
        avail = new_avail

    # --- Bilanz prüfen ---
    report = {}
    for commodity, dem in total_demand.items():
        sup = avail[commodity]
        report[commodity] = {"max_supply": sup, "demand": dem}
        if sup + tol < dem:
            problems.append(
                f"Commodity '{commodity}': fixe Nachfrage von {dem:.4g} "
                f"übersteigt die maximal bereitstellbare Menge von {sup:.4g} "
                f"(inkl. Conversion-Ketten)."
            )
    return problems, report


problems, report = check_commodity_balance(esM)
print(report)
problems

{'hydrogen': {'max_supply': np.float64(7.0), 'demand': 8.0}}


["Commodity 'hydrogen': fixe Nachfrage von 8 übersteigt die maximal bereitstellbare Menge von 7 (inkl. Conversion-Ketten)."]

In [289]:
def _island_map(esM, trans_edges):
    """Für jede Commodity: Mapping Region -> frozenset(Insel).
    Regionen ohne Transmission für die Commodity bilden Einzel-Inseln."""
    edges_by_commodity = defaultdict(set)
    for c, l1, l2 in trans_edges:
        edges_by_commodity[c].add((l1, l2))

    islands = {}
    for c in set(edges_by_commodity) | {
        comp.commodity
        for m in esM.componentModelingDict.values()
        for comp in m.componentsDict.values()
        if hasattr(comp, "commodity")
    }:
        # Union-Find light über BFS
        remaining = set(esM.locations)
        comp_map = {}
        adj = defaultdict(set)
        for l1, l2 in edges_by_commodity.get(c, ()):
            adj[l1].add(l2)
            adj[l2].add(l1)
        while remaining:
            start = remaining.pop()
            group, stack = {start}, [start]
            while stack:
                for nb in adj[stack.pop()]:
                    if nb in remaining:
                        remaining.discard(nb)
                        group.add(nb)
                        stack.append(nb)
            fz = frozenset(group)
            for loc in group:
                comp_map[loc] = fz
        islands[c] = comp_map
    return islands


def _collect_trans_edges(esM):
    """(commodity, loc1, loc2) für alle nutzbaren Transmission-Kanten."""
    edges = []
    transModel = esM.componentModelingDict.get("TransmissionModel")
    if transModel:
        for comp in transModel.componentsDict.values():
            elig = _unwrap_ip(getattr(comp, "processedLocationalEligibility", None))
            if elig is None:
                elig = _unwrap_ip(getattr(comp, "locationalEligibility", None))
            if elig is not None:
                pairs = [_parse_edge(e, esM.locations)
                         for e, v in elig.items() if v > 0]
            else:
                locs = list(esM.locations)
                pairs = [(a, b) for a in locs for b in locs if a != b]
            for l1, l2 in pairs:
                edges.append((comp.commodity, l1, l2))
    return edges


def check_timestep_balance(esM, tol=1e-6, max_iter=50):
    """Prüft für jeden Zeitschritt und jede Transmission-Insel, ob die fixe
    Nachfrage jeder Commodity durch Quellen, Speicher (Entladeschranke) und
    Conversion-Ketten gedeckt werden kann.

    Obere Schranke -> notwendige, nicht hinreichende Bedingung.
    Ignoriert: Speicher-Ladehistorie, Input-Konkurrenz zwischen Conversions,
    Kapazitätsgrenzen einzelner Transmission-Leitungen (nur Topologie zählt).

    Gibt (problems, report) zurück; report[t][commodity][insel] =
    {'max_supply': ..., 'demand': ...} für Inseln mit Nachfrage.
    """
    problems = []
    report = {}
    h = esM.hoursPerTimeStep
    T = getattr(esM, "numberOfTimeSteps", None) or len(esM.totalTimeSteps)
    locs = sorted(esM.locations)

    srcSnkModel = esM.componentModelingDict.get("SourceSinkModel")
    convModel = esM.componentModelingDict.get("ConversionModel")
    storModel = esM.componentModelingDict.get("StorageModel")

    sources, sinks = [], []
    if srcSnkModel:
        for comp in srcSnkModel.componentsDict.values():
            (sources if comp.sign == 1 else sinks).append(comp)
    conversions = list(convModel.componentsDict.values()) if convModel else []
    storages = list(storModel.componentsDict.values()) if storModel else []

    trans_edges = _collect_trans_edges(esM)
    islands = _island_map(esM, trans_edges)

    # --- zeitunabhängige Vorarbeit ---
    src_data = []
    for comp in sources:
        src_data.append((
            comp,
            _get_time_series(comp, "operationRateFix"),
            _get_time_series(comp, "operationRateMax"),
            _capacity_series(comp, esM),
            _eligible_locations(comp, esM),
        ))

    stor_supply = defaultdict(float)  # (commodity, loc) -> max Entladung/Schritt
    for comp in storages:
        caps = _capacity_series(comp, esM)
        d_rate = float(getattr(comp, "dischargeRate", 1) or 1)
        d_eff = float(getattr(comp, "dischargeEfficiency", 1) or 1)
        for loc in _eligible_locations(comp, esM):
            cap = float(caps.get(loc, math.inf))
            stor_supply[(comp.commodity, loc)] += cap * d_rate * d_eff * h

    conv_data = []
    for comp in conversions:
        factors = comp.commodityConversionFactors
        if isinstance(factors, dict) and all(isinstance(k, int) for k in factors):
            factors = _unwrap_ip(factors)
        conv_data.append((
            comp,
            {c: abs(f) for c, f in factors.items() if f < 0},
            {c: f for c, f in factors.items() if f > 0},
            _capacity_series(comp, esM),
            _eligible_locations(comp, esM),
        ))

    snk_data = [(s, _get_time_series(s, "operationRateFix")) for s in sinks]

    # --- Schleife über Zeitschritte ---
    for t in range(T):
        # 1) lokales Angebot je (commodity, region): Quellen + Speicher
        local = defaultdict(float)
        for comp, rfix, rmax, caps, elig in src_data:
            rate = rfix if rfix is not None else rmax
            for loc in elig:
                if comp.hasCapacityVariable:
                    cap = float(caps.get(loc, math.inf))
                    r = float(rate[loc].iloc[t]) if (rate is not None and loc in rate.columns) else 1.0
                    local[(comp.commodity, loc)] += 0.0 if r == 0 else cap * r * h
                else:
                    if rate is not None and loc in rate.columns:
                        local[(comp.commodity, loc)] += float(rate[loc].iloc[t])
                    else:
                        local[(comp.commodity, loc)] = math.inf
        for key, val in stor_supply.items():
            local[key] += val

        # 2) Fixpunkt: Conversions produzieren, Inseln poolen
        conv_out = defaultdict(float)
        for _ in range(max_iter):
            # Pool je (commodity, Insel)
            pool = defaultdict(float)
            for (c, loc), val in local.items():
                pool[(c, islands[c][loc])] += val
            for (c, loc), val in conv_out.items():
                pool[(c, islands[c][loc])] += val

            new_out = defaultdict(float)
            for comp, inputs, outputs, caps, elig in conv_data:
                for loc in elig:
                    cap = float(caps.get(loc, math.inf))
                    op = math.inf if math.isinf(cap) else cap * h
                    for c_in, f_in in inputs.items():
                        # Zugriff auf den Insel-Pool des Inputs, ohne eigenen Output
                        avail_in = pool[(c_in, islands[c_in][loc])]
                        op = min(op, avail_in / f_in)
                    if op > 0:
                        for c_out, f_out in outputs.items():
                            new_out[(c_out, loc)] += op * f_out

            keys = set(conv_out) | set(new_out)
            if all(math.isclose(conv_out[k], new_out[k], rel_tol=1e-9, abs_tol=tol)
                   or (math.isinf(conv_out[k]) and math.isinf(new_out[k]))
                   for k in keys):
                conv_out = new_out
                break
            conv_out = new_out

        # finaler Pool für diesen Zeitschritt
        pool = defaultdict(float)
        for (c, loc), val in local.items():
            pool[(c, islands[c][loc])] += val
        for (c, loc), val in conv_out.items():
            pool[(c, islands[c][loc])] += val

        # 3) Nachfrage je (commodity, Insel) gegen Pool prüfen
        demand = defaultdict(float)
        for snk, dem in snk_data:
            if dem is None:
                continue
            for loc in dem.columns:
                d = float(dem[loc].iloc[t])
                if d > 0:
                    demand[(snk.commodity, islands[snk.commodity][loc])] += d

        report_t = defaultdict(dict)
        for (c, isl), dem_val in demand.items():
            sup = pool[(c, isl)]
            report_t[c][isl] = {"max_supply": sup, "demand": dem_val}
            if sup + tol < dem_val:
                problems.append(
                    f"t={t}: Commodity '{c}' in Insel {sorted(isl)}: "
                    f"Nachfrage {dem_val:.4g} > max. Angebot {sup:.4g}."
                )
        report[t] = dict(report_t)

    return problems, report


problems, report = check_timestep_balance(esM)
problems

check_timestep_balance(esM)

([],
 {0: {'hydrogen': {frozenset({'Region1',
               'Region2',
               'Region3'}): {'max_supply': 12.285, 'demand': 1.0}}},
  1: {'hydrogen': {frozenset({'Region1',
               'Region2',
               'Region3'}): {'max_supply': 12.285, 'demand': 1.0}}},
  2: {'hydrogen': {frozenset({'Region1',
               'Region2',
               'Region3'}): {'max_supply': 10.184999999999999,
     'demand': 1.0}}},
  3: {'hydrogen': {frozenset({'Region1',
               'Region2',
               'Region3'}): {'max_supply': 10.184999999999999,
     'demand': 5.0}}}})